# 0. Environment and Configuration

In [ ]:
import subprocess
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import os
import workflows
import json
import glob
from plotnine import *

In [ ]:
e4s_cl_path = '/lus/bnchlu1/neth/e4s-cl/install/bin'
go_dir = '/lus/scratch/crickett/go/'
apptainer_bin_path = '/lus/scratch/crickett/software/usr/bin'
apptainer_cache_dir = '/lus/bnchlu1/neth/.local/apptainer/'

workflow_dir = os.getcwd()
workflows.set_workflow_log(f'{workflow_dir}/ampere-workflow.log')

# 1. Download and Install Ampere

In [ ]:
os.chdir(workflow_dir)
if not os.path.exists('ampere-repo'):
    workflows.run_cmd('git clone https://github.com/adam-mcdaniel/ampere ampere-repo')

os.chdir('ampere-repo')
workflows.run_cmd('pip install .')
os.chdir('..')
import ampere

# 2. Prepare and Launch Arkouda Server

## Clone Setup Repository and Create e4s-cl Profile

In [ ]:
workflows.cd(workflow_dir)
if not os.path.exists('e4s-cl-setup'):
    workflows.run_cmd('git clone https://github.com/arezaii/e4s-cl-setup.git')

output = subprocess.run('e4s-cl profile list', shell=True, check=True, stdout=subprocess.PIPE).stdout.decode('utf-8')
if 'arkouda-container' not in output:
    
    workflows.cd('e4s-cl-setup')
    input = 'y\ny\nn\narkouda-container\n'
    subprocess.run('./setup_all.sh', shell=True, check=True, input=input.encode('utf-8'))
    workflows.cd(workflow_dir)

workflows.run_cmd('e4s-cl profile select arkouda-container')


## Download Arkouda Container

In [ ]:
container_path = os.path.join(workflow_dir, 'chapel-arkouda.sif')

if not os.path.exists(container_path):
    workflows.run_cmd(f'apptainer pull {container_path} docker://arezaiihpe/chapel-2.6.0-arkouda-2025.12.16:latest')

workflows.run_cmd(f'e4s-cl profile edit --image {container_path}')


## Launch and Connect Arkouda Server



In [ ]:
try:
    import arkouda as ak
except ImportError:
    workflows.cd(workflow_dir)
    # clone with the tag v2025.12.16
    if not os.path.exists('arkouda-repo'):
        workflows.run_cmd('git clone --branch v2025.12.16 https://github.com/Bears-R-Us/arkouda.git arkouda-repo')

    workflows.cd('arkouda-repo')
    workflows.run_cmd('pip install .')
    workflows.cd(workflow_dir)
import arkouda as ak



In [ ]:
def find_server_name_and_port():
    # First check if there's a slurm job from me with the name "arkouda-server"
    result = subprocess.run('squeue --me --name arkouda-server', shell=True, check=True, stdout=subprocess.PIPE).stdout.decode('utf-8')
    lines = result.strip().split('\n')
    if len(lines) > 1:
        # If there is, extract the job ID and find the node it's running on
        job_id = lines[1].split()[0]
        result = subprocess.run(f'scontrol show job {job_id}', shell=True, check=True, stdout=subprocess.PIPE).stdout.decode('utf-8')
    else:
        print("No slurm job found with name 'arkouda-server'")
        return None

    # Now read the output file for the arkouda server to find the URL
    output_file = f'arkouda_arkouda-server_{job_id}.out'

    # match for *                          server listening on tcp://x1002c2s4b0n0:5555                          *
    pattern = re.compile(r'server listening on (tcp://\S+)')
    with open(output_file, 'r') as f:
        for line in f:
            match = pattern.search(line)
            if match:
                url = match.group(1)
                name,port = url.split('//')[1].split(':')
                return name, port
    print("Could not find arkouda server URL in output file")
    return None

In [ ]:
if find_server_name_and_port() is None:
    workflows.run_cmd('e4s-cl-setup/bin/launch_arkouda.sh -N 2')
find_server_name_and_port()
#ak.connect(connect_url=find_server_url())


# 3. Prepare Ampere

In [ ]:
from ampere import Ensemble, MetricConfig, MetricType, AmpereSession, Visualizer, connect

host, port = find_server_name_and_port()
connect(server=host, port=port)

# Define Metric Configs
configs = {
    re.compile(r".*rocm.*energy.*"): MetricConfig(MetricType.CUMULATIVE, scale_factor=1e-6),
    re.compile(r".*rocm.*power.*"): MetricConfig(MetricType.INSTANTANEOUS, scale_factor=1e-6),
}

# Define Topology Resolver
def my_hpc_topology(metric_name, ranks):
    # Each of our GPU devices is shared by 2 MPI ranks, so we map metric names to the corresponding ranks
    if 'device=4' in metric_name: return [r for r in ranks if r.name in ['MPI Rank 0', 'MPI Rank 1']]
    if 'device=2' in metric_name: return [r for r in ranks if r.name in ['MPI Rank 2', 'MPI Rank 3']]
    if 'device=6' in metric_name: return [r for r in ranks if r.name in ['MPI Rank 4', 'MPI Rank 5']]
    if 'device=0' in metric_name: return [r for r in ranks if r.name in ['MPI Rank 6', 'MPI Rank 7']]
    return ranks

# Load Traces
ranks = [f"MPI Rank {i}" for i in range(8)]
topo = {"Node0": ranks}
ensemble = Ensemble.from_trace_paths(glob.glob('/lus/bnchlu1/neth/fastotf2/workflows/scaling/out/locgroup_dist_block_PARQUET_8nodes_64threads_nosort_trial0/*.parquet'), topo, configs)



In [ ]:
df_joules = ak.DataFrame.concat([
    ensemble.attribute(
        metric,
        topology_resolver=my_hpc_topology,
        strategy='exclusive'
    ) for metric in [
        "A2rocm_smi:::energy_count:device=0",
        "A2rocm_smi:::energy_count:device=2",
        "A2rocm_smi:::energy_count:device=4",
        "A2rocm_smi:::energy_count:device=6"]
])
print("Attribution complete.")